# Chapter 2 Practical 01: Feature Vectors and Similarity

Learning objectives:
- Represent movies with numeric, categorical, and multi-label features.
- Compute cosine similarity, Jaccard similarity, and Euclidean distance.
- Interpret what each similarity measure means.
- Plot items in a simple 2D feature space.

Slide connection: item representation, feature vectors, vector space, cosine similarity, Jaccard similarity, and Euclidean distance.


We start with five familiar movies and a few hand-written content features. Small examples make the vector idea easier to see.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from sklearn.preprocessing import MultiLabelBinarizer, MinMaxScaler

movies = pd.DataFrame({
    "title": ["Inception", "Interstellar", "Titanic", "The Matrix", "Toy Story"],
    "duration_min": [148, 169, 195, 136, 81],
    "rating": [8.8, 8.7, 7.9, 8.7, 8.3],
    "genres": [
        ["Sci-Fi", "Thriller", "Action"],
        ["Sci-Fi", "Adventure", "Drama"],
        ["Romance", "Drama"],
        ["Sci-Fi", "Action"],
        ["Animation", "Adventure", "Comedy", "Family"],
    ],
})
movies


Numeric features need scaling because duration and rating live on different ranges.


In [ ]:
scaler = MinMaxScaler()
numeric_features = pd.DataFrame(
    scaler.fit_transform(movies[["duration_min", "rating"]]),
    columns=["duration_scaled", "rating_scaled"],
    index=movies["title"],
)
numeric_features.round(2)


Genres are multi-label categorical features: one movie can belong to several genres. We encode each genre as a 0/1 column.


In [ ]:
mlb = MultiLabelBinarizer()
genre_features = pd.DataFrame(
    mlb.fit_transform(movies["genres"]),
    columns=mlb.classes_,
    index=movies["title"],
)
genre_features


Now we combine numeric and genre features into one item-feature matrix.


In [ ]:
feature_matrix = pd.concat([numeric_features, genre_features], axis=1)
feature_matrix.round(2)


Cosine similarity compares the angle between vectors. It is often useful when the pattern of features matters more than the raw size of the vector.


In [ ]:
cosine = pd.DataFrame(
    cosine_similarity(feature_matrix),
    index=feature_matrix.index,
    columns=feature_matrix.index,
)
cosine.round(2)


Jaccard similarity compares overlap between sets. Here we use it only for genre sets.


In [ ]:
def jaccard(a, b):
    a, b = set(a), set(b)
    return len(a & b) / len(a | b)

jaccard_rows = []
for other in movies["title"]:
    base_genres = movies.loc[movies["title"] == "Inception", "genres"].iloc[0]
    other_genres = movies.loc[movies["title"] == other, "genres"].iloc[0]
    jaccard_rows.append({"movie": other, "jaccard_with_inception": jaccard(base_genres, other_genres)})

pd.DataFrame(jaccard_rows).sort_values("jaccard_with_inception", ascending=False)


Euclidean distance measures straight-line distance. Smaller means more similar, so we sort ascending.


In [ ]:
distances = pd.DataFrame(
    euclidean_distances(feature_matrix),
    index=feature_matrix.index,
    columns=feature_matrix.index,
)
distances["Inception"].rename("distance_from_inception").sort_values().round(2)


A 2D plot cannot show all features, but it helps students see the intuition of distance in a feature space.


In [ ]:
plot_df = feature_matrix[["Sci-Fi", "Romance"]].copy()
plot_df["title"] = plot_df.index

ax = plot_df.plot.scatter(x="Sci-Fi", y="Romance", s=120, figsize=(6, 4))
for _, row in plot_df.iterrows():
    ax.text(row["Sci-Fi"] + 0.02, row["Romance"] + 0.02, row["title"])
ax.set_title("Movies in a tiny 2D genre space")
ax.set_xlim(-0.1, 1.25)
ax.set_ylim(-0.1, 1.25)
plt.show()


## What did we learn?

- Feature vectors turn item metadata into numbers.
- One-hot and multi-hot encoding make categorical features usable.
- Cosine, Jaccard, and Euclidean measures answer related but different similarity questions.

Exercises:
1. Add one more movie and recompute all three similarities.
2. Change the numeric scaling or remove numeric features. Which recommendations change?
